In [ ]:
# ==========================================
# TESIS ZMVM: PM, clima y salud
# Autor: Arely Leal
# Descripción: Script en Python para generar grafico de las concentraciones horarias por año para material particulado. 
# FUNCIONA PARA PM10 (2000-2019) Y PM2.5 (2003-2019)
# ==========================================


In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

# =========================
# CARGAR DATOS
# =========================
archivo = "AGREGAR RUTA DEL ARCHIVO"
df = pd.read_csv(archivo)

df["FECHA"] = pd.to_datetime(df["FECHA"], errors="coerce")

# MODIFICACIÓN 1: Limpiar nulos en fecha y forzar AÑO a entero
df = df.dropna(subset=["FECHA"]) 
df["AÑO"] = df["FECHA"].dt.year.astype(int)

df["HORA"] = pd.to_numeric(df["HORA"], errors="coerce")

municipios = [col for col in df.columns if col not in ["FECHA","HORA","AÑO"]]

for col in municipios:
    df[col] = pd.to_numeric(df[col], errors="coerce")

carpeta = "AGREGAR RUTA DE SALIDA"
os.makedirs(carpeta, exist_ok=True)

# =========================
# DICCIONARIO DE ACENTOS: MODIFICAR CON BASE A LA DISPONIBILIDAD DE DATOS PARA PM10 O PM2.25
# =========================
correccion_nombres = {
"ACOLMAN":"Acolman",
"ALVARO_OBREGON":"Álvaro Obregón",
"AZCAPOTZALCO":"Azcapotzalco",
"BENITO_JUAREZ":"Benito Juárez",
"CHALCO":"Chalco",
"COACALCO DE BERRIOZABAL":"Coacalco de Berriozábal",
"COYOACAN":"Coyoacán",
"CUAJIMALPA_DE_MORELOS":"Cuajimalpa de Morelos",
"CUAUHTEMOC":"Cuauhtémoc",
"ECATEPEC_DE_MORELOS":"Ecatepec de Morelos",
"GUSTAVO_A_MADERO":"Gustavo A. Madero",
"IZTACALCO":"Iztacalco",
"IZTAPALAPA":"Iztapalapa",
"MIGUEL_HIDALGO":"Miguel Hidalgo",
"MILPA_ALTA":"Milpa Alta",
"NAUCALPAN_DE_JUAREZ":"Naucalpan de Juárez",
"NEZAHUALCOYOTL":"Nezahualcóyotl",
"OCOYOACAC":"Ocoyoacac",
"TEPOTZOTLAN":"Tepotzotlán",
"TLAHUAC":"Tláhuac",
"TLALNEPANTLA": "Tlalnepantla de Baz",
"TLALPAN":"Tlalpan",
"TULTITLAN":"Tultitlán",
"VENUSTIANO_CARRANZA":"Venustiano Carranza"
}

# =========================
# COLORES POR AÑO
# =========================
color_anio = {
2003:"#1f77b4", 2004:"#ff7f0e", 2005:"#2ca02c", 2006:"#9467bd",
2007:"#8c564b", 2008:"#e377c2", 2009:"#7f7f7f", 2010:"#bcbd22",
2011:"#17becf", 2012:"#393b79", 2013:"#637939", 2014:"#8c6d31",
2015:"#843c39", 2016:"#7b4173", 2017:"#3182bd", 2018:"#31a354",
2019:"#d62728"
}

# Asegurar que la lista de años sean enteros
anios = sorted(df["AÑO"].unique().astype(int))

# =========================
# CALCULAR PROMEDIOS
# =========================
todos_resultados=[]

for municipio in municipios:
    for anio in anios:
        df_anio=df[df["AÑO"]==anio].copy()
        total_dias=df_anio["FECHA"].dt.date.nunique()

        for hora in range(1,25):
            datos=df_anio[df_anio["HORA"]==hora][municipio]
            n_validos=datos.notna().sum()
            porcentaje=n_validos/total_dias if total_dias>0 else np.nan

            if porcentaje>=0.75:
                promedio=datos.mean()
            else:
                promedio=np.nan

            todos_resultados.append({
                "MUNICIPIO":municipio,
                "AÑO":anio,
                "HORA":hora,
                "PROMEDIO":promedio
            })

df_promedios=pd.DataFrame(todos_resultados)

# =========================
# ESCALA GLOBAL
# =========================
valores=df_promedios["PROMEDIO"].dropna()
y_min=0
y_max=np.ceil(valores.max())
y_max=5*np.ceil(y_max/5)

print(f"Escala vertical unificada: {y_min} a {y_max}")

# =========================
# GRAFICAR
# =========================
for municipio in municipios:
    df_local=df_promedios[df_promedios["MUNICIPIO"]==municipio].copy()
    plt.figure(figsize=(16,6))

    handles=[]
    labels=[]

    for anio in anios:
        datos_anio=df_local[df_local["AÑO"]==anio].copy()
        datos_anio=datos_anio.sort_values("HORA")

        datos_anio["PROMEDIO"]=datos_anio["PROMEDIO"].interpolate(
            method="linear",
            limit_direction="both"
        )

        if datos_anio["PROMEDIO"].notna().sum()>0:
            linea,=plt.plot(
                datos_anio["HORA"],
                datos_anio["PROMEDIO"],
                color=color_anio.get(anio,"#333333"),
                linewidth=1.2,
                alpha=0.95,
                label=str(int(anio)) # Aseguramos conversión a string de entero
            )
            handles.append(linea)
            labels.append(str(int(anio)))

    # LÍNEAS DE REFERENCIA: MODIFICAR EN FUNCIÓN DE PM10 O PM2.5
    nom=plt.axhline(75, color="red", linestyle="--", linewidth=2, label="Límite 24hrs NOM 2014 (75 µg/m³)")
    oms=plt.axhline(45, color="orange", linestyle="--", linewidth=2, label="Límite 24hrs OMS 2021 (45 µg/m³)")

    handles.extend([nom,oms])

    #MODIFICAR EN FUNCIÓN DE PM10 O PM2.5
    labels.extend(["Límite 24hrs NOM 2014 (75 µg/m³)", "Límite 24hrs OMS 2021 (45 µg/m³)"])

    # TÍTULO
    titulo=correccion_nombres.get(municipio.upper(),municipio.replace("_"," ").title())
    plt.title(titulo.upper(), fontsize=14, fontweight="bold", pad=35)
    plt.text(12.5, y_max*1.03, "Promedio por hora PM..", ha="center", fontsize=12)

    # EJES
    plt.xlabel("Hora del día",fontsize=12,fontweight="bold",labelpad=20)
    plt.ylabel("Concentración (µg/m³)",fontsize=12,fontweight="bold",labelpad=15)
    plt.xticks(range(1,25))
    plt.xlim(1,24)
    plt.ylim(y_min,y_max)

    # LEYENDA
    plt.legend(handles, labels, loc="lower center", bbox_to_anchor=(0.5,-0.45), ncol=5, frameon=False)
    plt.grid(True)

    for spine in plt.gca().spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.4)
        spine.set_color("gray")

    plt.tight_layout(rect=[0,0.5,1,0.93])
    nombre=municipio.replace(" ","_").replace("/","_")
    plt.savefig(f"{carpeta}/{nombre}_pm_promedio_hora.png", dpi=300, bbox_inches="tight")
    plt.close()

print("Gráficas generadas correctamente.")
print(f"Carpeta de salida: {carpeta}")

/var/folders/8f/xcm8kn2d7csg_55l0rmqbcjc0000gn/T/ipykernel_5769/1272548563.py:12: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["FECHA"] = pd.to_datetime(df["FECHA"], errors="coerce")


Escala vertical unificada: 0 a 120.0


/var/folders/8f/xcm8kn2d7csg_55l0rmqbcjc0000gn/T/ipykernel_5769/1272548563.py:170: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout(rect=[0,0.5,1,0.93])


Gráficas generadas correctamente.
Carpeta de salida: /Users/arelyleal/Downloads/TESIS/GRAFICAS_PM10_HORARIAS_ESTILO_TESIS_FINAL
